In [1]:
import subprocess, os
os.makedirs("data", exist_ok=True)
subprocess.run(["python3", "dataset.py", "--out", "data/spam_dataset.csv"], check=True)
print("Dataset ready.")

Dataset ready.


In [2]:
%%writefile train_and_export_model.py
import argparse
import joblib
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--data", default="data/spam_dataset.csv")
    parser.add_argument("--out", default="data/model.joblib")
    args = parser.parse_args()

    df = pd.read_csv(args.data)
    X = df["text"]
    y = df["label"]
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

    model = make_pipeline(TfidfVectorizer(), MultinomialNB())
    model.fit(X_train, y_train)

    preds = model.predict(X_test)
    print(f"accuracy={accuracy_score(y_test, preds):.4f}  f1={f1_score(y_test, preds, pos_label='spam'):.4f}")

    joblib.dump({"model": model, "feature_columns": ["text"]}, args.out)
    print(f"Saved model bundle to {args.out}")


if __name__ == "__main__":
    main()

Overwriting train_and_export_model.py


In [3]:
subprocess.run(["python3", "train_and_export_model.py"], check=True)
print(os.path.getsize("data/model.joblib"), "bytes")

accuracy=1.0000  f1=1.0000
Saved model bundle to data/model.joblib
5934 bytes


In [ ]:
%%writefile predictor_app.py
"""
predictor_app.py — AI Operations (AIOps), Module 3 Lecture 2b
FastAPI predictor implementing the KServe V1 inference protocol.
"""
import os, socket, time
import joblib
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel

MODEL_PATH = os.environ.get("MODEL_PATH", "data/model.joblib")
POD_NAME = os.environ.get("POD_NAME", socket.gethostname())
NODE_NAME = os.environ.get("NODE_NAME", "unknown")

app = FastAPI(title="Email Classifier Predictor")
_bundle = None


@app.on_event("startup")
def load_model():
    global _bundle
    _bundle = joblib.load(MODEL_PATH)
    print(f"Loaded model ({len(_bundle['feature_columns'])} features) on pod={POD_NAME} node={NODE_NAME}")


class PredictRequest(BaseModel):
    instances: list[str] 


@app.get("/healthz")
def healthz():
    return {"status": "ok", "pod": POD_NAME, 'version' : 'new',"node": NODE_NAME}

 

@app.post("/v1/models/{model_name}:predict")
def predict(model_name: str, request: PredictRequest):
    if _bundle is None:
        raise HTTPException(status_code=503, detail="Model not loaded yet")
    
    model = _bundle["model"]  

    t0 = time.time()
    predictions = model.predict(request.instances).tolist()
    latency_ms = round((time.time() - t0) * 1000, 2)

    return {
        "predictions": predictions,
        "served_by_pod": POD_NAME,
        "served_by_node": NODE_NAME,
        "latency_ms": latency_ms,
    }

Overwriting predictor_app.py


In [5]:
%%writefile k8s-manifest.yaml

apiVersion: apps/v1
kind: Deployment
metadata:
  name: email-classifier-deployment
  labels:
    app: email-classifier
spec:
  replicas: 2
  selector:
    matchLabels:
      app: email-classifier
  template:
    metadata:
      labels:
        app: email-classifier
    spec:
      containers:
      - name: predictor
        image: docker.io/library/email-classifier:v1
        imagePullPolicy: Never
        ports:
        - containerPort: 8000
        resources:
          requests:
            cpu: "100m"
            memory: "128Mi"
          limits:
            cpu: "500m"
            memory: "512Mi"
        env:
        - name: MODEL_PATH
          value: "data/model.joblib"
        - name: POD_NAME
          valueFrom:
            fieldRef:
              fieldPath: metadata.name
        - name: NODE_NAME
          valueFrom:
            fieldRef:
              fieldPath: spec.nodeName
        readinessProbe:
          httpGet:
            path: /healthz
            port: 8000
          initialDelaySeconds: 5
          periodSeconds: 5
---
apiVersion: v1
kind: Service
metadata:
  name: email-classifier-service
spec:
  type: ClusterIP
  selector:
    app: email-classifier
  ports:
  - protocol: TCP
    port: 80
    targetPort: 8000

Overwriting k8s-manifest.yaml


In [6]:
import subprocess, time, requests, os

env = os.environ.copy()
env.update({"MODEL_PATH": "data/model.joblib", "POD_NAME": "local-test", "NODE_NAME": "local"})

proc = subprocess.Popen(
    ["uvicorn", "predictor_app:app", "--host", "0.0.0.0", "--port", "28080"],
    env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)